In [13]:
import pandas as pd
import re
import os
import string
from bs4 import BeautifulSoup

In [10]:
# โหลดข้อมูลจากไฟล์ CSV
file_path = 'job_data_merged_1.csv'
data = pd.read_csv(file_path)

# 1. ตรวจสอบข้อมูลเบื้องต้น
print("ข้อมูลเบื้องต้น:")
print(data.head())  # ดู 5 แถวแรก
print(data.info())  # ดูข้อมูลเกี่ยวกับประเภทของคอลัมน์และข้อมูลที่หายไป

ข้อมูลเบื้องต้น:
   Unnamed: 0          Category Workplace  \
0           0  Business Analyst    Remote   
1           1  Business Analyst    Remote   
2           2  Business Analyst   On-site   
3           3  Business Analyst   On-site   
4           4  Business Analyst    Remote   

                                       Location             Department  \
0                                United Kingdom             Operations   
1             Makati, Metro Manila, Philippines                 Aux HQ   
2  Al-Dajeej, Al Farwaniyah Governorate, Kuwait       PWC Technologies   
3               London, England, United Kingdom  Consultants, Advisory   
4                                United Kingdom             Operations   

        Type  
0  Full time  
1  Full time  
2  Full time  
3  Full time  
4  Full time  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------

In [21]:
# 2. ตรวจสอบข้อมูลที่หายไป
missing_data = data.isnull().sum()
print("\nข้อมูลที่หายไป:")
print(missing_data)


ข้อมูลที่หายไป:
Category            0
Workplace          56
Location           68
Department        166
Type              169
final_features      0
dtype: int64


In [33]:
# Text cleaning
# ---------------------------
def super_clean(text):
    # กรณี missing หรือค่าว่าง
    if pd.isna(text) or str(text).strip() == "":
        return "unknown"

    text = str(text).lower()

    # เก็บ a-z, 0-9, +, # และช่องว่าง
    text = re.sub(r'[^a-z0-9+#\s]', ' ', text)

    # ลบช่องว่างเกิน
    text = " ".join(text.split())

    # ถ้า clean แล้วกลายเป็นค่าว่าง
    if text == "":
        return "unknown"

    return text


# ---------------------------
# Main processing
# ---------------------------
def process_and_save(
    input_file="job_data_merged_1.csv",
    output_file="cleaned_job_data.csv"
):
    if not os.path.exists(input_file):
        print(f"❌ ไม่พบไฟล์: {input_file}")
        return None

    print("--- เริ่มการ Clean ข้อมูล ---")
    df = pd.read_csv(input_file)

    # ลบคอลัมน์ขยะพวก Unnamed ถ้ามี
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", na=False)]

    target_cols = ['Category', 'Workplace', 'Location', 'Department', 'Type']

    # 1) ถ้าคอลัมน์ไหนไม่มีในไฟล์ ให้สร้างขึ้นมาเลย
    for col in target_cols:
        if col not in df.columns:
            print(f"⚠️ ไม่พบคอลัมน์ {col} -> สร้างคอลัมน์ใหม่เป็น 'unknown'")
            df[col] = "unknown"

    # 2) เติม missing value ในคอลัมน์ต้นฉบับก่อน
    df[target_cols] = df[target_cols].fillna("unknown")

    # 3) แปลงค่าว่าง '' หรือช่องว่างล้วน ให้เป็น unknown
    for col in target_cols:
        df[col] = df[col].apply(
            lambda x: "unknown" if str(x).strip() == "" else x
        )

    # 4) Clean ทีละคอลัมน์
    for col in target_cols:
        df[f'cleaned_{col}'] = df[col].apply(super_clean)

    # 5) รวมทุก cleaned column เป็น final_features
    cleaned_cols = [f'cleaned_{col}' for col in target_cols]
    df['final_features'] = df[cleaned_cols].agg(' '.join, axis=1)

    # 6) ลบช่องว่างเกินอีกรอบ
    df['final_features'] = df['final_features'].apply(lambda x: " ".join(str(x).split()))

    # 7) กันกรณี final_features ว่างจริง ๆ
    df['final_features'] = df['final_features'].replace("", "unknown")

    # เลือกคอลัมน์ที่จะเซฟ
    cols_to_save = target_cols + ['final_features']
    df[cols_to_save].to_csv(output_file, index=False)

    print(f"✅ Clean สำเร็จ! บันทึกไปที่: {output_file}")
    print("\nจำนวน missing หลัง clean:")
    print(df[cols_to_save].isnull().sum())

    print("\nตัวอย่างข้อมูลที่ได้ (5 แถวแรก):")
    print(df[cols_to_save].head())

    return df


if __name__ == "__main__":
    process_and_save()

--- เริ่มการ Clean ข้อมูล ---
✅ Clean สำเร็จ! บันทึกไปที่: cleaned_job_data.csv

จำนวน missing หลัง clean:
Category          0
Workplace         0
Location          0
Department        0
Type              0
final_features    0
dtype: int64

ตัวอย่างข้อมูลที่ได้ (5 แถวแรก):
           Category Workplace                                      Location  \
0  Business Analyst    Remote                                United Kingdom   
1  Business Analyst    Remote             Makati, Metro Manila, Philippines   
2  Business Analyst   On-site  Al-Dajeej, Al Farwaniyah Governorate, Kuwait   
3  Business Analyst   On-site               London, England, United Kingdom   
4  Business Analyst    Remote                                United Kingdom   

              Department       Type  \
0             Operations  Full time   
1                 Aux HQ  Full time   
2       PWC Technologies  Full time   
3  Consultants, Advisory  Full time   
4             Operations  Full time   

                

In [34]:
# โหลดไฟล์ที่ clean แล้วมาเช็ค
check_df = pd.read_csv("cleaned_job_data.csv")

# 1. เช็คตัวพิมพ์ใหญ่ (ต้องได้ 0)
upper_exists = check_df['final_features'].str.contains(r'[A-Z]').sum()
print(f"จำนวนแถวที่มีตัวพิมพ์ใหญ่: {upper_exists}")

# 2. เช็คเครื่องหมายคอมม่า (ต้องได้ 0)
comma_exists = check_df['final_features'].str.contains(',').sum()
print(f"จำนวนแถวที่มีเครื่องหมายคอมม่า: {comma_exists}")

จำนวนแถวที่มีตัวพิมพ์ใหญ่: 0
จำนวนแถวที่มีเครื่องหมายคอมม่า: 0
